# KMeans clustering in OMLE

Train, evaluate, export, inspect, validate, visualize, and verify a complete fitted model. Run all cells from a fresh kernel.

## Dataset

**[Synthetic blobs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html)** — Generate 600 two-dimensional samples from three Gaussian centers with seed 42. Known generating labels are used only for external evaluation, never for fitting the clustering pipeline. No external dataset is downloaded or redistributed.

Data is loaded directly through scikit-learn; no dataset directory is required. The loader description below preserves additional source details and attribution.

In [ ]:
import json
import platform
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans

import omle
import omle_runtime as omr
from omle_convert import export_omle

SEED = 42
ATOL = 1e-4
RTOL = 1e-4
packages = ['numpy', 'pandas', 'scikit-learn', 'omle', 'omle-convert', 'omle-runtime', 'omle-viewer']
environment = {"python": platform.python_version(), "packages": {p: version(p) for p in packages}}
display(environment)

## Load and split

Reserve a fixed 20% test split. Fit the model and any preprocessing on training rows only. No hyperparameter tuning uses the held-out test set.

In [ ]:
features, generating_labels = make_blobs(
    n_samples=600, n_features=2, centers=3, cluster_std=1.0, random_state=SEED
)
X = pd.DataFrame(features, columns=["feature_0", "feature_1"])
y = pd.Series(generating_labels, name="generating_cluster")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Training rows: {len(X_train)}; test rows: {len(X_test)}")
display(X_train.head())

## Train and evaluate

Clustering is fitted without labels. Silhouette measures separation on held-out scaled features; adjusted Rand index compares assignments with generating labels and ignores label numbering.

In [ ]:
pipeline = Pipeline([("scaler", StandardScaler()), ("clusterer", KMeans(n_clusters=3, n_init=10, random_state=SEED))])
pipeline.fit(X_train)
native_labels = pipeline.predict(X_test)

In [ ]:
scaled_test = pipeline.named_steps["scaler"].transform(X_test)
metrics = {
    "silhouette": float(silhouette_score(scaled_test, native_labels)),
    "adjusted_rand_index": float(adjusted_rand_score(y_test, native_labels)),
}
display(pd.Series(metrics, name="held_out_metrics"))
display(pd.Series(native_labels).value_counts().sort_index().rename("test_cluster_counts"))

## Export the fitted pipeline

Export the entire fitted pipeline, including preprocessing when present. Embedded verification examples use training rows; the independent parity check uses every held-out row. Rerunning overwrites this recipe’s own generated artifacts.

In [ ]:
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents]
                  if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Start Jupyter from omle-models or a notebook subdirectory.")
output_dir = repo_root / "models" / "clustering" / "blobs"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "sklearn-kmeans.omle"
export_omle(
    pipeline, str(model_path), X=X_train,
    model_name="blobs-sklearn-kmeans", model_version="0.1.0",
    random_state=SEED, n_verify=10, n_sample=3, verify_atol=ATOL, verify_rtol=RTOL,
)
print(f"Exported {model_path.relative_to(repo_root)} ({model_path.stat().st_size:,} bytes)")

## Inspect the exported model

Inspect declared inputs, outputs, and operators before running structural validation. A `-1` shape dimension denotes a variable batch size.

In [ ]:
import omle

exported_model = omle.load(str(model_path))

print("Model inputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
    }
    for spec in exported_model.inputs
]))

print("Model outputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
        "role": spec.role.name,
    }
    for spec in exported_model.outputs
]))

### Operators included in the model

Show the actual graph operators and their imported namespace versions.

In [ ]:
operator_versions = {
    item.namespace: item.version for item in exported_model.operator_imports
}
operators = pd.DataFrame([
    {
        "node": node.name,
        "namespace": node.domain,
        "operator": node.op,
        "version": operator_versions.get(node.domain),
    }
    for node in exported_model.nodes
])
display(operators)

### Validate the model

Structural validation checks graph and operator declarations. Errors stop execution; warnings remain visible. Numeric prediction parity is checked separately below.

In [ ]:
validation = omle.validate(exported_model)
print(validation)
if not validation.is_valid:
    raise ValueError(f"Exported OMLE model is invalid:\n{validation}")

### Explore the interactive graph

Select the graph nodes to inspect their fitted parameters and connections. The viewer is embedded locally; no model upload or external viewer server is needed.

Run the cell below in JupyterLab or your IDE notebook editor to interact with the graph. GitHub’s static notebook preview will not run it.

In [ ]:
from omle_viewer import show

viewer = show(exported_model, height=650)

## Verify OMLE inference

Run the artifact on raw test features in their original column order. Compare with this model’s native implementation, not a separately trained model from another framework. Cluster numbering is required to match this exported fitted model exactly, even though independent clustering fits can permute cluster IDs.

In [ ]:
runtime_model = omr.load(str(model_path))
omle_labels = runtime_model.predict(X_test)
assert omle_labels.shape == native_labels.shape
assert np.isfinite(omle_labels).all()
np.testing.assert_array_equal(omle_labels, native_labels)
parity = {"test_rows": len(X_test), "label_agreement": float(np.mean(omle_labels == native_labels))}
parity.update({"atol": ATOL, "rtol": RTOL,
               "native_prediction_dtype": str(native_labels.dtype),
               "omle_prediction_dtype": str(omle_labels.dtype)})
display(pd.Series(parity, name="conversion_fidelity"))
print("PASS: OMLE predictions match the native fitted model.")

## Save the run record

Save exact dependency versions, dataset identity, feature order, training configuration, split membership, quality metrics, and conversion fidelity next to this model. Generated artifacts remain ignored by Git. A fixed seed alone does not guarantee identical fits across dependency versions.

In [ ]:
report = {
    "environment": environment,
    "dataset": "sklearn.datasets.make_blobs", "dataset_source": "https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html",
    "seed": SEED, "test_size": 0.2,
    "feature_names": list(X.columns),
    "train_indices": X_train.index.tolist(), "test_indices": X_test.index.tolist(),
    "model_parameters": {str(k): str(v) for k, v in pipeline.get_params(deep=True).items()},
    "metrics": metrics, "parity": parity,
}
report["generator_parameters"] = {"n_samples": 600, "n_features": 2, "centers": 3, "cluster_std": 1.0, "random_state": SEED}
report_path = output_dir / "sklearn-kmeans-environment.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Saved {report_path.relative_to(repo_root)}")